## ***IMPORT LIBRERIE***


In [ ]:
import os
import pandas as pd
import numpy as np
from app_config import REPORTS_DIR, FIGURES_DIR, MODELS_DIR
import sliding_window_on_data
from torch.utils.data import DataLoader
import glob

from models.DeepConvLSTM import DeepConvLSTM, HARDataset 
import optuna
from optuna.pruners import MedianPruner
import optuna.visualization as vis
import train
import torch
import torch.nn as nn
import train_with_cm
import matplotlib.pyplot as plt
#definisco il path da cui leggere i .csv

from utils.log_config import logger
from figures import plot_CM

#definisco il path da cui leggere i .csv

path='C:\codes\HumanActivityRecognition\data\apdd_data'
logger.debug(path)


<>:25: SyntaxWarning: invalid escape sequence '\c'
<>:25: SyntaxWarning: invalid escape sequence '\c'
C:\Users\n.laporta.inst\AppData\Local\Temp\ipykernel_25088\3123876836.py:25: SyntaxWarning: invalid escape sequence '\c'
  path='C:\codes\HumanActivityRecognition\data\apdd_data'
2025-06-12 16:02:22.353 | INFO     | app_config:<module>:11 - PROJ_ROOT path is: C:\codes\HumanActivityRecognition
2025-06-12 16:02:32,989 - INFO - myapp - Logging configured from C:\codes\HumanActivityRecognition\HumanActivityRecognition\utils\base_config.json
2025-06-12 16:02:42,700 - INFO - myapp - GPU not available, training on CPU.
c:\Users\nicolo.laporta\OneDrive - SUPSI\Desktop\virtual_envs\har_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-06-12 16:02:43,599 - DEBUG - matplotlib - matplotlib data path: c:\Users\nicolo.l

## ***DATA PREPROCESSING***

*scansiono recording e filtro per righe non nulle*  
*OUTPUT: unico dataframe con tutte le attività non nulle*

In [ ]:
#Visto che l'informazione relativa all'id del bambino lo abbiamo, così come abbiamo anche l'informazione relativa
#al giocattolo, vado a filtrare i .csv in modo tale da avere solo le righe che hanno l'attività non nulla e poi
#appendo tutte le righe delle righe non nulle in un unico dataframe
#per tutti i file che terminano in .csv nella cartella path
# Definisco il percorso della cartella contenente i CSV

# Nome del file CSV finale
final_csv_path = os.path.join(path, 'df_BA_non_null.csv')

# Controllo se il file esiste già
if os.path.exists(final_csv_path):
    logger.debug(f"Il file {final_csv_path} esiste già. Lo sto caricando...")
    df_ball = pd.read_csv(final_csv_path)
else:
    #trovo tutti i file che corrispondono a "BA" nella cartella path e li stampo a schermo
    files = glob.glob(os.path.join(path, "*_BA*.csv"))
    logger.debug("Files:", files)
    
    kid_ball, kid_ball_no_null = [], [] # liste per salvare utenti prima e dopo il merge 
    df_list_ball = [] # lista vuota per appendere i dataframe con attività non nulla

    for file in files:
        df = pd.read_csv(file)
        
        logger.debug("Original shape:", df.shape)
        kid_ball.append(df['kid_id'].unique())
        df = df[df['action_id'] != 0] #filtro le righe con action_id non nullo
        logger.debug("Filtered shape:", df.shape)
        kid_ball_no_null.append(df['kid_id'].unique())

        logger.debug("Columns:", df.columns)
        logger.debug("Action counts:\n", df['action'].value_counts())
        logger.debug("Toy counts:\n", df['toy_id'].value_counts())
        logger.debug("="*50)  # Separatore tra i file

    # mi stampo gli utenti prima di fare il merge e dopo il merge
    if len(kid_ball) > 0:
        logger.info(f"Numero di utenti che hanno fatto almeno un azione: {len(kid_ball_no_null)/len(kid_ball)}")
    else:
        logger.info("Nessun utente trovato nei file analizzati (kid_ball è vuoto).")

    '''
    Visto che l'informazione relativa all'id del bambino lo abbiamo, così come abbiamo anche l'informazione relativa
    al giocattolo, vado a filtrare i .csv in modo tale da avere solo le righe che hanno l'attività non nulla e poi
    appendo tutte le righe non nulle in un unico dataframe per tutti i file che terminano in .csv nella balltella path
    '''

    for file in os.listdir(path):
        if not file.endswith('.csv'):
            continue
    
        #leggo solo i file che dopo l'undescore ha BA*.csv
        if file.split('_')[-1].startswith('BA') and file.endswith('.csv'): #controllo che il file termini con .csv
            df_temp=pd.read_csv(os.path.join(path,file))
            df_temp = df_temp[df_temp['action_id'] != 0]
            df_list_ball.append(df_temp)

    df_ball = pd.concat(df_list_ball)
    logger.info("Dimensioni del df_ball con tutte le attività non nulle")
    logger.info(df_ball.shape)
    logger.info(df_ball.columns)
    logger.info(df_ball['action'].value_counts())

    # salvo il dataframe
    df_ball.to_csv(os.path.join(path,'df_BA_non_null.csv'),index=False)

--- Logging error ---
Traceback (most recent call last):
  File "C:\Program Files\Python312\Lib\logging\__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "C:\Program Files\Python312\Lib\logging\__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "C:\Program Files\Python312\Lib\logging\__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "C:\Program Files\Python312\Lib\logging\__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\nicolo.laporta\OneDrive - SUPSI\Desktop\virtual_envs\har_venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c

ZeroDivisionError: division by zero

*per ogni attività trovata, salvo un .csv*  
*OUTPUT: un .csv per ogni attività non nulla (df_ball_action_11.csv, df_ball_action_19.csv ecc... )*


In [6]:
#ora divido il dataframe in base all'attività (action_id) e salvo i dataframe in un file .csv
#per ogni attività

for action_id in df_ball['action_id'].unique():
    df_action = df_ball[df_ball["action_id"] == action_id] #filtro il dataframe in base all'attività
    logger.debug(f"Dimensioni del dataframe df_ball_action_{action_id} - {df_action.shape}") #log delle dimensioni del dataframe
    logger.info(f"Conteggio delle attività per df_ball_action_{action_id}") #log del conteggio delle attività
    #salvo il dataframe
    df_action.to_csv(os.path.join(path,f'df_ball_action_{action_id}.csv'),index=False) #index=False per non salvare l'indice
    logger.debug(f"Salvato il dataframe df_ball_action_{action_id}.csv")

2025-05-09 10:34:55,146 - DEBUG - myapp - Dimensioni del dataframe df_ball_action_21 - (665, 25)
2025-05-09 10:34:55,149 - INFO - myapp - Conteggio delle attività per df_ball_action_21
2025-05-09 10:34:55,183 - DEBUG - myapp - Salvato il dataframe df_ball_action_21.csv
2025-05-09 10:34:55,188 - DEBUG - myapp - Dimensioni del dataframe df_ball_action_19 - (180, 25)
2025-05-09 10:34:55,191 - INFO - myapp - Conteggio delle attività per df_ball_action_19
2025-05-09 10:34:55,206 - DEBUG - myapp - Salvato il dataframe df_ball_action_19.csv
2025-05-09 10:34:55,212 - DEBUG - myapp - Dimensioni del dataframe df_ball_action_11 - (420, 25)
2025-05-09 10:34:55,213 - INFO - myapp - Conteggio delle attività per df_ball_action_11
2025-05-09 10:34:55,238 - DEBUG - myapp - Salvato il dataframe df_ball_action_11.csv
2025-05-09 10:34:55,243 - DEBUG - myapp - Dimensioni del dataframe df_ball_action_41 - (248, 25)
2025-05-09 10:34:55,244 - INFO - myapp - Conteggio delle attività per df_ball_action_41
2025-

## ***DATA PROCESSING***

*SLIDING WINDOW*  
*OUTPUT: unica matrice con tutte le windows concatenate*

In [4]:
#applico sliding window con la funzion process_csv
nb_sensor_channels = 9
sliding_window_length = 100
sliding_window_step = 50

#ora applico la funzione sliding window (che mi da come output x_window e y_window) a tutti i .csv relativi al giocattolo ball
#e poi concateno tutto in un unica x e y 

X, Y = [], []
kid_action_counts = {}

for action_file in [f for f in os.listdir(path) if f.endswith('.csv') and f.split('_')[1] == 'ball']:
    file_path = os.path.join(path, action_file)
    action = action_file.split('_')[-1].split('.')[0]

    X_windows, Y_windows, kid_id_action_dict = sliding_window_on_data.process_csv(file_path, nb_sensor_channels, sliding_window_length, sliding_window_step)

    X.append(X_windows)
    Y.append(Y_windows)


    logger.info(f"Numero  totale di finestre per l'azione {action}:{len(X_windows)}")
    kid_action_counts[f"ball_action_{action}"] = kid_id_action_dict #aggiungo il dizionario al dizionario principale per tenere traccia del numero di finestre per ogni bambino per ogni azione, ogni ball_action è una chiave e il valore è un dizionario con il numero di finestre per ogni bambino
    logger.info(f"Contenuto finale di kid_action_counts: {kid_action_counts}") #per veere quante finestre per ogni azione e per ogni bambino sono state elaborte 


# Concateno tutti i dati in un unico array per X e Y
X = np.concatenate(X, axis=0)
Y = np.concatenate(Y, axis=0)


# Stampo le dimensioni di X e Y
logger.info(f"Dimensioni di X finale: {X.shape}")
logger.info(f"Dimensioni di Y finale: {Y.shape}")

2025-05-09 10:33:31,312 - DEBUG - myapp - sto leggendo il file csv: C:\codes\HumanActivityRecognition\data\pdd_data\df_ball_action_11.csv
2025-05-09 10:33:31,327 - INFO - myapp - Kid_id: 3002, X_kid shape: (420, 9), Y_kid shape: (420,)
2025-05-09 10:33:31,330 - INFO - myapp - Numero di finestre estratte: 7
2025-05-09 10:33:31,330 - DEBUG - myapp - Campioni rimanenti dopo l'ultima finestra completa: 20
2025-05-09 10:33:31,333 - INFO - myapp - Padding estremo applicato. Padding code: 2
2025-05-09 10:33:31,337 - DEBUG - myapp - Nuova finestra con padding: (1, 100, 9)
2025-05-09 10:33:31,340 - INFO - myapp - Numero totale di finestre (dopo padding finale): 8
2025-05-09 10:33:31,341 - DEBUG - myapp - X_windows shape after sliding window: (8, 100, 9)
2025-05-09 10:33:31,342 - DEBUG - myapp - Padding codes: [2]
2025-05-09 10:33:31,342 - INFO - myapp - Numero di finestre estratte: 7
2025-05-09 10:33:31,344 - DEBUG - myapp - Campioni rimanenti dopo l'ultima finestra completa: 20
2025-05-09 10:3

>>> Kid_ids: [3002]
>>> Kid_ids: [3002]
>>> Kid_ids: [3002]


2025-05-09 10:33:31,517 - INFO - myapp - Kid_id: 3002, X_kid shape: (248, 9), Y_kid shape: (248,)
2025-05-09 10:33:31,520 - INFO - myapp - Numero di finestre estratte: 3
2025-05-09 10:33:31,521 - DEBUG - myapp - Campioni rimanenti dopo l'ultima finestra completa: 48
2025-05-09 10:33:31,522 - INFO - myapp - Padding normale applicato. Padding code: 1
2025-05-09 10:33:31,524 - DEBUG - myapp - Nuova finestra con padding: (1, 100, 9)
2025-05-09 10:33:31,527 - INFO - myapp - Numero totale di finestre (dopo padding finale): 4
2025-05-09 10:33:31,529 - DEBUG - myapp - X_windows shape after sliding window: (4, 100, 9)
2025-05-09 10:33:31,530 - DEBUG - myapp - Padding codes: [1]
2025-05-09 10:33:31,532 - INFO - myapp - Numero di finestre estratte: 3
2025-05-09 10:33:31,533 - DEBUG - myapp - Campioni rimanenti dopo l'ultima finestra completa: 48
2025-05-09 10:33:31,534 - INFO - myapp - Padding normale applicato. Padding code: 1
2025-05-09 10:33:31,535 - DEBUG - myapp - Nuova finestra con padding:

>>> Kid_ids: [3002]


*SPLIT DATASET IN TRS, VS,TS*  
*OUTPUT: array NumPy di TRS/VS/TS (X e Y)*

In [ ]:
from collections import defaultdict
split_ratio_train = 0.7
split_ratio_val = 0.15
split_ratio_test = 0.15

X_train, Y_train = [], []
X_val, Y_val = [], []
X_test, Y_test = [], []


#trovo le azioni uniche
unique_actions = np.unique(Y)
logger.info("Azioni uniche: %s", unique_actions)

#Liste per gli indici delle finestre
train_indices_totali = []
val_indices_totali = []
test_indices_totali = []


for action in unique_actions:
    # Trovo gli indici delle finestre corrispondenti a ciascuna azione
    action_indices = np.where(Y == action)[0]
    num_windows = len(action_indices)
    logger.info("Azione %s: %d finestre", action, num_windows)
    
    # Gestione casi speciali in base al numero di finestre disponibili
    if num_windows <= 2:
        # Se ci sono solo 1 o 2 finestre, tutte vanno nel training
        train_indices = action_indices
        val_indices = np.array([], dtype=int)
        test_indices = np.array([], dtype=int)
    elif num_windows == 3:
        # Per 3 finestre: 1 train, 1 val, 1 test [1,1,1]
        train_indices = action_indices[:1]
        val_indices = action_indices[1:2]
        test_indices = action_indices[2:]
    elif num_windows == 4:
        # Per 4 finestre: 2 train, 1 val, 1 test [2,1,1]
        train_indices = action_indices[:2]
        val_indices = action_indices[2:3]
        test_indices = action_indices[3:]
    elif num_windows == 5:
        # Per 5 finestre: 3 train, 1 val, 1 test [3,1,1]
        train_indices = action_indices[:3]
        val_indices = action_indices[3:4]
        test_indices = action_indices[4:]
    elif num_windows == 6:
        # Per 6 finestre: 4 train, 1 val, 1 test [4,1,1]
        train_indices = action_indices[:4]
        val_indices = action_indices[4:5]
        test_indices = action_indices[5:]
    else:
        # Per numero di finestre >= 7, uso i rapporti standard
        num_train = int(num_windows * split_ratio_train)
        num_test = int(num_windows * split_ratio_test)
        num_val = num_windows - num_train - num_test
    
        # Divido gli indici delle finestre in train val e test
        train_indices = action_indices[:num_train]
        val_indices = action_indices[num_train:num_train + num_val]
        test_indices = action_indices[num_train + num_val:]
    

    logger.info("Azione %s - Train: %d, Val: %d, Test: %d", 
               action, len(train_indices), len(val_indices), len(test_indices))

    
    # Aggiungo le finestre al train al val e al test
    X_train.append(X[train_indices])
    Y_train.append(Y[train_indices])
    X_val.append(X[val_indices])
    Y_val.append(Y[val_indices])
    X_test.append(X[test_indices])
    Y_test.append(Y[test_indices])

    # Accumulo gli indici
    train_indices_totali.extend(train_indices.tolist())
    val_indices_totali.extend(val_indices.tolist())
    test_indices_totali.extend(test_indices.tolist())

# Concateno i dati in un unico array per X e Y
# Concateno tutti i dati
X_train = np.concatenate(X_train, axis=0)
Y_train = np.concatenate(Y_train, axis=0)
X_val = np.concatenate(X_val, axis=0)
Y_val = np.concatenate(Y_val, axis=0)
X_test = np.concatenate(X_test, axis=0)
Y_test = np.concatenate(Y_test, axis=0)

# Flatten etichette se necessario
Y_train = Y_train.flatten()
Y_val = Y_val.flatten()
Y_test = Y_test.flatten()

logger.info("Train shape: %s %s", X_train.shape, Y_train.shape)
logger.info("Val shape: %s %s", X_val.shape, Y_val.shape)
logger.info("Test shape: %s %s", X_test.shape, Y_test.shape)

# Conto quante finestre per ogni azione in train e test
train_action_counts = defaultdict(list)
test_action_counts = defaultdict(list)

for i, label in enumerate(Y_train):
    train_action_counts[label].append(i)

for i, label in enumerate(Y_test):
    test_action_counts[label].append(i)

# Liste per nuovi dati aggiornati
X_train_new, Y_train_new = list(X_train), list(Y_train)
X_test_new, Y_test_new = list(X_test), list(Y_test)


azioni_modificate = 0

for action in np.unique(Y):
    train_ids = train_action_counts.get(action, []) #Indici delle finestre nel train mi servono per vedere se ho almeno 3 finestre
    test_ids = test_action_counts.get(action, []) #Indici delle finestre nel test mi servono per vedere se ho almeno 1 finestra

    if len(test_ids) == 1 and len(train_ids) >= 3:
        idx_to_move = train_ids[-1]  # prendo l'ultima finestra di quella azione nel train

        logger.debug(f"Azione {action} - Sposto la finestra con indice originale {idx_to_move} dal train al test")
        logger.debug(f"Y_train[idx]: {Y_train_new[idx_to_move]}")
        logger.debug(f"X_train[idx][:5]: {X_train_new[idx_to_move][:5]}")  # primi 5 campioni della finestra
        logger.debug(f"Shape della finestra: {X_train_new[idx_to_move].shape}")

        # Aggiungo l'indice della finestra spostata al test
        test_indices_totali.append(train_indices_totali[train_ids[-1]])

        # Rimuovo l'indice dalla lista degli indici di train
        train_indices_totali.remove(train_indices_totali[train_ids[-1]])

        # Sposto la finestra nel test
        X_test_new.append(X_train_new[idx_to_move])
        Y_test_new.append(Y_train_new[idx_to_move])

        #verifico che la finestra sia stata spostata correttamente
        logger.debug(f"Y_test[idx]: {Y_test_new[-1]}")
        logger.debug(f"X_test[idx][:5]: {X_test_new[-1][:5]}")  # primi 5 campioni della finestra
        logger.debug(f"Shape della finestra: {X_test_new[-1].shape}")

        # La rimuovo dal train
        del X_train_new[idx_to_move]
        del Y_train_new[idx_to_move]

        azioni_modificate += 1
        logger.info(f"Azione {action}: spostata una finestra da train a test per bilanciare meglio")

# Converto di nuovo in numpy
X_train = np.array(X_train_new)
Y_train = np.array(Y_train_new)
X_test = np.array(X_test_new)
Y_test = np.array(Y_test_new)

logger.info("Azioni modificate: %d", azioni_modificate)


# Salvo gli indici finali dopo il bilanciamento
np.savez(f"{path}\\split_final_indices_ball.npz",
         train=np.array(train_indices_totali),
         val=np.array(val_indices_totali),
         test=np.array(test_indices_totali))

# Stampa controllo
logger.info("Train shape: %s %s", X_train.shape, Y_train.shape)
logger.info("Val shape: %s %s", X_val.shape, Y_val.shape)
logger.info("Test shape: %s %s", X_test.shape, Y_test.shape)




2025-05-09 10:55:16,709 - INFO - myapp - Azioni uniche: [11. 19. 21. 41.]
2025-05-09 10:55:16,806 - INFO - myapp - Azione 11.0: 8 finestre
2025-05-09 10:55:16,817 - INFO - myapp - Azione 11.0 - Train: 5, Val: 2, Test: 1
2025-05-09 10:55:16,822 - INFO - myapp - Azione 19.0: 3 finestre
2025-05-09 10:55:16,826 - INFO - myapp - Azione 19.0 - Train: 1, Val: 1, Test: 1
2025-05-09 10:55:16,832 - INFO - myapp - Azione 21.0: 13 finestre
2025-05-09 10:55:16,836 - INFO - myapp - Azione 21.0 - Train: 9, Val: 3, Test: 1
2025-05-09 10:55:16,840 - INFO - myapp - Azione 41.0: 4 finestre
2025-05-09 10:55:16,843 - INFO - myapp - Azione 41.0 - Train: 2, Val: 1, Test: 1
2025-05-09 10:55:16,854 - INFO - myapp - Train shape: (17, 100, 9) (17,)
2025-05-09 10:55:16,858 - INFO - myapp - Val shape: (7, 100, 9) (7,)
2025-05-09 10:55:16,865 - INFO - myapp - Test shape: (4, 100, 9) (4,)
2025-05-09 10:55:16,872 - DEBUG - myapp - Azione 11.0 - Sposto la finestra con indice originale 4 dal train al test
2025-05-09 10

In [8]:
# Ricarico gli indici salvati
loaded_data = np.load(f"{path}\\split_final_indices_ball.npz")

# Estraggo gli indici delle finestre
train_indices_totali = loaded_data['train']
val_indices_totali = loaded_data['val']
test_indices_totali = loaded_data['test']

# Stampa per verificare
logger.info(f"Train indices: {train_indices_totali.shape}")
logger.info(f"Val indices: {val_indices_totali.shape}")
logger.info(f"Test indices: {test_indices_totali.shape}")

2025-05-09 10:55:22,049 - INFO - myapp - Train indices: (15,)
2025-05-09 10:55:22,057 - INFO - myapp - Val indices: (7,)
2025-05-09 10:55:22,057 - INFO - myapp - Test indices: (6,)


*RIASSEGNAZIONE DELLE ETICHETTE*

In [9]:
# Trova tutte le etichette uniche presenti nei dati
unique_labels = np.unique(Y_train)

# Crea un dizionario che mappa ogni etichetta originale a un valore consecutivo
label_mapping = {label: idx for idx, label in enumerate(unique_labels)}

# Stampa il dizionario per vedere il mapping
logger.info("Mapping delle etichette: %s", label_mapping)

# Applica il mapping ai dataset di train e test
Y_train_mapped = np.array([label_mapping[y] for y in Y_train])
Y_test_mapped = np.array([label_mapping[y] for y in Y_test])
Y_val_mapped = np.array([label_mapping[y] for y in Y_val])

# Controllo finale
logger.info("Nuove etichette train: %s", np.unique(Y_train_mapped))
logger.info("Nuove etichette val: %s ", np.unique(Y_val_mapped))
logger.info("Nuove etichette test: %s", np.unique(Y_test_mapped))




# Creo i dataset per il training val e test
train_dataset = HARDataset(X_train, Y_train_mapped)
val_dataset = HARDataset(X_val, Y_val_mapped)
test_dataset = HARDataset(X_test, Y_test_mapped)

 #stampo il dataset di training e di test a livello di dimensioni
logger.info("Lunghezza dataset di training: %s", len(train_dataset))
logger.info("Lunghezza dataset di validazione: %s", len(val_dataset))
logger.info("Lunghezza dataset di test: %s", len(test_dataset))


logger.debug(f"Tipo di train_dataset: {type(train_dataset)}")
logger.debug(f"Tipo di val_dataset: {type(val_dataset)}")
logger.debug(f"Tipo di test_dataset: {type(test_dataset)}")



2025-05-09 10:55:23,684 - INFO - myapp - Mapping delle etichette: {np.float64(11.0): 0, np.float64(19.0): 1, np.float64(21.0): 2, np.float64(41.0): 3}
2025-05-09 10:55:23,699 - INFO - myapp - Nuove etichette train: [0 1 2 3]
2025-05-09 10:55:23,705 - INFO - myapp - Nuove etichette val: [0 1 2 3] 
2025-05-09 10:55:23,709 - INFO - myapp - Nuove etichette test: [0 1 2 3]
2025-05-09 10:55:23,917 - INFO - myapp - Lunghezza dataset di training: 15
2025-05-09 10:55:23,925 - INFO - myapp - Lunghezza dataset di validazione: 7
2025-05-09 10:55:23,929 - INFO - myapp - Lunghezza dataset di test: 6
2025-05-09 10:55:23,933 - DEBUG - myapp - Tipo di train_dataset: <class 'models.DeepConvLSTM.HARDataset'>
2025-05-09 10:55:23,933 - DEBUG - myapp - Tipo di val_dataset: <class 'models.DeepConvLSTM.HARDataset'>
2025-05-09 10:55:23,938 - DEBUG - myapp - Tipo di test_dataset: <class 'models.DeepConvLSTM.HARDataset'>


## ***OPTUNA***

*nella funzione obiettivo: uso train e val*  
*stampo cm di train e val*  
*train finale con dati di test con cm*




In [ ]:
BEST_MODEL_PATH = os.path.join(MODELS_DIR, "best_model_inference_ball_without_anything.pkl")
BEST_SCORE_PATH = os.path.join(REPORTS_DIR, "best_score_inference_ball_without_anything.txt")

best_hyperparams_file = os.path.join(REPORTS_DIR, 'best_hyperparameters_inference_ball_without_anything.csv')
if os.path.exists(best_hyperparams_file):
    logger.debug(f"Carico i migliori iperparametri da {best_hyperparams_file}")
    best_hyperparameters=pd.read_csv(best_hyperparams_file).iloc[0].to_dict()
    best_lr = best_hyperparameters['lr']
    best_batch_size = int(best_hyperparameters['batch_size'])

else:
    def objective(trial):
        # DefiniscO gli iperparametri da ottimizzare
        lr = trial.suggest_float('lr', 1e-4, 1e-1, log=True)
        batch_size = trial.suggest_categorical('batch_size', [2,4,6])

        # Creo i DataLoader con il batch_size suggerito
        
        train_loader = DataLoader(train_dataset, batch_size=batch_size,shuffle=True, drop_last=True)

        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, drop_last=True)

        # Creo il modello con gli iperparametri suggeriti e rimuovo la testa originale, in modo da non caricare i pesi associati
        model = DeepConvLSTM()

    
        model.load_state_dict(torch.load(r'C:\codes\HumanActivityRecognition\models\best_model_dl_without_anything.pkl', map_location=torch.device('cpu')), strict=False)



        # Ora sostituisco la testa del modello con la nuova dimensione di classi (4)
        num_ftrs = model.fc.in_features
        model.fc = nn.Linear(num_ftrs, 4)  # 4 classi
        model.set_n_classes(4)


        # Congelo tutti i parametri tranne quelli della testa (fully connected)
        for param in model.parameters():
            param.requires_grad = False  # Congelo tutti i pesi

        # Sblocco i parametri della testa (fully connected)
        for param in model.fc.parameters():
            param.requires_grad = True  # Solo i pesi della testa saranno addestrabili

        # Eseguo l'allenamento
        best_f1_score = train_with_cm.train(model, train_loader, val_loader, epochs=100, batch_size=batch_size, lr=lr)

        # Salvo modello se è il migliore finora
        is_better = False

        if not os.path.exists(BEST_SCORE_PATH):
            is_better = True
        else:
            with open(BEST_SCORE_PATH, "r") as f:
                best_score_so_far = float(f.read())
            if best_f1_score > best_score_so_far:
                is_better = True

        if is_better:
            torch.save(model.state_dict(), BEST_MODEL_PATH)  # use joblib.dump() if it's not a PyTorch model
            with open(BEST_SCORE_PATH, "w") as f:
                f.write(str(best_f1_score))
            logger.info(f"Nuovo miglior modello salvato in: {BEST_MODEL_PATH} con F1 score: {best_f1_score}")

        return best_f1_score

    # Creazione studio Optuna ottimizza, nel senso di minimizzare la loss in 100 prove
    # Creazione dello studio con il MedianPruner
    study = optuna.create_study(
        direction='maximize', 
        pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=10)  # Parametri di pruning per evitare di continuare trial non promettenti: n_startup_trials=5 significa che i primi 5 trial non verranno prunati, n_warmup_steps=10 significa che dopo 10 trial verrà applicato il pruning
        #Il pruner interromperà automaticamente i trial che non sono promettenti, basandosi sui punteggi parziali (F1-score) ottenuti durante l'allenamento.
    )
    study.optimize(objective, n_trials=100)

    logger.info("Best hyperparameters: ", study.best_params)
    logger.info("Highest F1-score: ", study.best_value)

    #salvo i best hyperparameters
    best_hyperparameters = study.best_params
    best_hyperparameters['best_f1_score'] = study.best_value
    best_hyperparameters_df = pd.DataFrame([best_hyperparameters])
    best_hyperparameters_df.to_csv(os.path.join(REPORTS_DIR, 'best_hyperparameters_inference_without_norm_without_sampler_aug_4_classes.csv'), index=False)

    best_lr = study.best_params['lr']
    best_batch_size = study.best_params['batch_size']

    #visualizzare la storia dell'ottimizzazione effettuata da Optuna. Ci permette di vedere come l'f1 score
    # è cambiato nel corso delle diverse prove (trials) durante l'ottimizzazione.
    file_name = "optimization_history_dl_without_norm_without_sampler_aug_4_classes.png"
    fig=vis.plot_optimization_history(study)
    plt.show()

    # Salvo il grafico nella cartella FIGURES con il nome specificato
    fig.write_image(os.path.join(FIGURES_DIR, file_name))

    logger.debug(f"Grafico salvato in figures /{file_name}")

#TO DO: ALTRO SCRIPT
#STAMPO CM DELL'ALLENAMENTO SUL TRAINING SET
plot_CM(
    mdl_class=DeepConvLSTM,
    mdl_weights=BEST_MODEL_PATH,
    X= X_train,
    Y=Y_train_mapped,
    batch_size=best_batch_size,
    figure_name="cm_train_best_hyp_inference_optuna_dl_without_norm_without_sampler_aug_4_classes"
)

#STAMPO CM DELL'ALLENAMENTO SUL VAL SET
plot_CM(
    mdl_class=DeepConvLSTM,
    mdl_weights=BEST_MODEL_PATH,
    X=X_val,
    Y=Y_val_mapped,
    batch_size=best_batch_size,
    figure_name="cm_val_best_hyp_inference_optuna_dl_without_norm_without_sampler_aug_4_classes"
)

# Creo modello e carico pesi del miglior modello (trovato prima in optuna)
model = DeepConvLSTM(n_classes=4)

# Rimuovo la testa originale, in modo da non caricare i pesi associati
model.load_state_dict(
    torch.load(
        BEST_MODEL_PATH,
        map_location=torch.device('cpu')
    ),
    strict=False
)


# Congelo tutti i parametri tranne quelli della testa (fully connected)
for param in model.parameters():
        param.requires_grad = False  # Congela tutti i pesi


for name, param in model.named_parameters():
    print(f"{name} requires_grad={param.requires_grad}")



# Creo i DataLoader con i migliori iperparametri
test_loader = DataLoader(test_dataset, batch_size=best_batch_size, drop_last=True, shuffle=False)



# Definisco la funzione di perdita
criterion = nn.CrossEntropyLoss()


# Eseguo l'eval sul test set usando i migliori iperparametri
test_loss, test_acc, test_f1 = train_with_cm.evaluate_model(model, test_loader, figure_name= "cm_test_best_hyp_optuna_inference_ball_without_norm_without_anything", save_confusion_matrix=True)



















2025-05-09 00:13:59,689 - DEBUG - myapp - Carico i migliori iperparametri da C:\codes\HumanActivityRecognition\reports\best_hyperparameters_inference_without_norm_without_sampler_aug_4_classes.csv
2025-05-09 00:13:59,696 - DEBUG - myapp - Number of classes: 4


2025-05-09 00:13:59,812 - INFO - myapp - Predictions saved to: C:\codes\HumanActivityRecognition\reports\cm_train_best_hyp_inference_optuna_dl_without_norm_without_sampler_aug_4_classes_predictions.csv
2025-05-09 00:13:59,870 - DEBUG - matplotlib.colorbar - locator: <matplotlib.ticker.AutoLocator object at 0x0000021093EF84C0>
2025-05-09 00:14:00,218 - INFO - myapp - Confusion Matrix saved to: C:\codes\HumanActivityRecognition\reports\figures\cm_train_best_hyp_inference_optuna_dl_without_norm_without_sampler_aug_4_classes_CM.png
2025-05-09 00:14:00,218 - DEBUG - myapp - Number of classes: 4
2025-05-09 00:14:00,274 - INFO - myapp - Predictions saved to: C:\codes\HumanActivityRecognition\reports\cm_val_best_hyp_inference_optuna_dl_without_norm_without_sampler_aug_4_classes_predictions.csv
2025-05-09 00:14:00,320 - DEBUG - matplotlib.colorbar - locator: <matplotlib.ticker.AutoLocator object at 0x0000021095A83880>
2025-05-09 00:14:00,652 - INFO - myapp - Confusion Matrix saved to: C:\codes\

conv1.weight requires_grad=False
conv1.bias requires_grad=False
conv2.weight requires_grad=False
conv2.bias requires_grad=False
conv3.weight requires_grad=False
conv3.bias requires_grad=False
conv4.weight requires_grad=False
conv4.bias requires_grad=False
lstm1.weight_ih_l0 requires_grad=False
lstm1.weight_hh_l0 requires_grad=False
lstm1.bias_ih_l0 requires_grad=False
lstm1.bias_hh_l0 requires_grad=False
lstm2.weight_ih_l0 requires_grad=False
lstm2.weight_hh_l0 requires_grad=False
lstm2.bias_ih_l0 requires_grad=False
lstm2.bias_hh_l0 requires_grad=False
fc.weight requires_grad=False
fc.bias requires_grad=False
